# 08 — Evaluation Exploration

Explores benchmark evaluation results — comparing model checkpoints across pipeline stages and model sizes.

Run after `make eval SIZE=125m`.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path
from datetime import datetime

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

RESULTS_DIR = Path('../results')
EVAL_DIR = RESULTS_DIR / 'eval'

## 1. Load evaluation results

In [ ]:
def load_latest_eval(model_name: str) -> dict | None:
    """Load the most recent eval result for a model."""
    model_dir = EVAL_DIR / model_name
    if not model_dir.exists():
        return None
    files = sorted(model_dir.glob('eval_*.json'))
    if not files:
        return None
    with open(files[-1]) as f:
        return json.load(f)

# Models to compare across pipeline stages
models = {
    'slm-125m (pretrain)':      'slm-125m',
    'slm-125m (chat SFT)':      'slm-125m-chat',
    'slm-125m (chat+code SFT)': 'slm-125m-chat-code',
    'slm-125m (DPO)':           'slm-125m-dpo',
}

results = {}
for label, model_name in models.items():
    data = load_latest_eval(model_name)
    if data:
        results[label] = data
        print(f"✓ Loaded: {label}")
    else:
        print(f"✗ Missing: {label} — run: make eval SIZE=125m")

## 2. Benchmark scores table

In [ ]:
BENCHMARKS = {
    'hellaswag':     ('hellaswag',      'acc_norm'),
    'arc_easy':      ('arc_easy',       'acc_norm'),
    'arc_challenge': ('arc_challenge',  'acc_norm'),
    'mmlu':          ('mmlu',           'acc'),
    'truthfulqa':    ('truthfulqa_mc2', 'acc'),
}

RANDOM_BASELINE = {
    'hellaswag': 0.25,
    'arc_easy': 0.25,
    'arc_challenge': 0.25,
    'mmlu': 0.25,
    'truthfulqa': 0.50,
}

def extract_score(eval_data: dict, task_key: str) -> float | None:
    task_name, metric = BENCHMARKS[task_key]
    task_results = eval_data.get('results', {})
    if task_name not in task_results:
        return None
    score = task_results[task_name].get(metric) or task_results[task_name].get(f'{metric},none')
    return round(score * 100, 1) if score else None

if results:
    rows = []
    for label, data in results.items():
        row = {'Model': label}
        for task_key in BENCHMARKS:
            score = extract_score(data, task_key)
            row[task_key] = f"{score:.1f}" if score else 'N/A'
        rows.append(row)

    # Add random baseline
    baseline = {'Model': 'Random baseline'}
    for task_key, rand in RANDOM_BASELINE.items():
        baseline[task_key] = f"{rand*100:.1f}"
    rows.insert(0, baseline)

    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
else:
    print("No evaluation results found — run: make eval SIZE=125m")

## 3. Benchmark scores chart

In [ ]:
if results:
    benchmarks = list(BENCHMARKS.keys())
    n_benchmarks = len(benchmarks)
    n_models = len(results)
    x = np.arange(n_benchmarks)
    width = 0.8 / max(n_models, 1)
    colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

    fig, ax = plt.subplots(figsize=(13, 5))

    # Random baseline
    for i, task_key in enumerate(benchmarks):
        ax.axhline(RANDOM_BASELINE[task_key] * 100, color='gray',
                   linestyle=':', alpha=0.5, linewidth=1)

    for j, (label, data) in enumerate(results.items()):
        scores = []
        for task_key in benchmarks:
            score = extract_score(data, task_key)
            scores.append(score or 0)
        offset = (j - n_models / 2 + 0.5) * width
        bars = ax.bar(x + offset, scores, width * 0.9,
                      label=label, color=colors[j % len(colors)], alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels([b.replace('_', ' ').title() for b in benchmarks])
    ax.set_ylabel('Score (%)')
    ax.set_title('SLM Benchmark Scores — Pipeline Stage Comparison', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9, loc='upper left')
    ax.set_ylim(0, 100)
    ax.text(0.99, 0.02, '· · · random baseline', transform=ax.transAxes,
            fontsize=8, color='gray', ha='right')

    plt.tight_layout()
    plt.show()

## 4. Improvement over baseline

In [ ]:
if results:
    print("Improvement over random baseline (percentage points):")
    print()

    header = f"{'Model':<30}" + "".join(f"  {b.replace('_',' '):<14}" for b in BENCHMARKS)
    print(header)
    print("-" * len(header))

    for label, data in results.items():
        row = f"  {label:<28}"
        for task_key in BENCHMARKS:
            score = extract_score(data, task_key)
            if score:
                delta = score - RANDOM_BASELINE[task_key] * 100
                row += f"  {delta:>+.1f}{'':9}"
            else:
                row += f"  {'N/A':<14}"
        print(row)

## 5. Cross-size comparison

In [ ]:
size_models = {
    '125M': 'slm-125m-dpo',
    '350M': 'slm-350m-dpo',
    '1B':   'slm-1b-dpo',
}

size_results = {}
for label, model_name in size_models.items():
    data = load_latest_eval(model_name)
    if data:
        size_results[label] = data

if size_results:
    benchmarks = list(BENCHMARKS.keys())
    x = np.arange(len(benchmarks))
    width = 0.25
    colors = ['#4C72B0', '#DD8452', '#55A868']

    fig, ax = plt.subplots(figsize=(13, 5))

    for j, (label, data) in enumerate(size_results.items()):
        scores = [extract_score(data, t) or 0 for t in benchmarks]
        offset = (j - 1) * width
        ax.bar(x + offset, scores, width * 0.9,
               label=label, color=colors[j], alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels([b.replace('_', ' ').title() for b in benchmarks])
    ax.set_ylabel('Score (%)')
    ax.set_title('SLM Benchmark Scores — Model Size Comparison (DPO)', fontsize=12, fontweight='bold')
    ax.legend(title='Model size', fontsize=9)
    ax.set_ylim(0, 100)
    plt.tight_layout()
    plt.show()
else:
    print("Cross-size results not available — run eval for 350m and 1b as well")

## 6. Radar chart — full model profile

In [ ]:
if results:
    categories = [b.replace('_', '\n') for b in BENCHMARKS]
    N = len(categories)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

    for j, (label, data) in enumerate(results.items()):
        values = [extract_score(data, t) or 0 for t in BENCHMARKS]
        values += values[:1]
        ax.plot(angles, values, 'o-', linewidth=2,
                color=colors[j % len(colors)], label=label, alpha=0.85)
        ax.fill(angles, values, alpha=0.1, color=colors[j % len(colors)])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=10)
    ax.set_ylim(0, 100)
    ax.set_title('SLM Model Profile — Radar Chart', fontsize=12,
                 fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
    plt.tight_layout()
    plt.show()